# pandas, and the rows that are not there

MichAl Academy, lesson 1.4.

Run each cell with **Shift+Enter**. pandas is already installed in Colab and
Kaggle.

The CSV is embedded in the notebook so there is no file to download. Everything
else behaves exactly as it would against a real file.

## 1. Load it, then check what you got

In [ ]:
import pandas as pd
from io import StringIO

CSV = '''host,tld,length,age_days,flagged
login.acme.co,co,13,1240,False
cdn.acme.co,co,11,980,False
acme-secure.tk,tk,14,3,True
mail.acme.co,co,12,1100,True
acme-login.tk,tk,13,,True
docs.acme.co,co,12,640,False
acme.verify.tk,tk,14,1,True
api.acme.co,co,11,1500,False
'''

df = pd.read_csv(StringIO(CSV))
df

In [ ]:
print("shape:", df.shape)
print()
print(df.dtypes)

Check `dtypes` every single time you load a file. pandas guessed each column's
type from the contents, and `age_days` came out as a float rather than an
integer because one value is empty.

Here is why that check matters. One stray value turns a whole column into text.

In [ ]:
DIRTY = '''host,ttl
a.example,300
b.example,300
c.example,unknown
'''

dirty = pd.read_csv(StringIO(DIRTY))
print(dirty.dtypes)

try:
    print(dirty[dirty["ttl"] > 100])
except TypeError as e:
    print("TypeError:", e)

`object` means text. Every numeric comparison against that column now fails, and
on a wider file you would not have noticed until it did.

## 2. Selecting

One name gives a Series. A list of names gives a DataFrame.

In [ ]:
print(type(df["tld"]).__name__)          # Series
print(type(df[["tld"]]).__name__)        # DataFrame

print()
print(df[["host", "flagged"]].head(3))

In [ ]:
# Rows by condition. The condition is itself a column of True and False.
print(df["flagged"].head(3))
print()
print(df[df["flagged"]])

`.loc` takes rows and columns together, which is the form you want whenever you
are about to write something.

In [ ]:
print(df.loc[df["tld"] == "tk", ["host", "age_days"]])

## 3. One question, four steps

Which top level domain is the problem?

In [ ]:
step1 = df[["tld", "flagged"]]
print("after picking columns:", step1.shape)

grouped = step1.groupby("tld")
print("rows per group:")
print(grouped.size())

`grouped` is not a table. It is the eight rows sorted into two buckets, waiting
to be told what to work out per bucket. That is why grouping is always two
calls.

In [ ]:
answer = df[["tld", "flagged"]].groupby("tld").agg(
    hosts=("flagged", "size"),
    flagged_rate=("flagged", "mean"),
)

print(answer)
print()
print("shape:", answer.shape, "  index:", list(answer.index))

Two rows out of eight. The shape says two columns while you can see three,
because `tld` became the index rather than a column. `answer.reset_index()`
turns it back into a column if you want it there.

## 4. The rows that are not there

`acme-login.tk` has no `age_days`. It is flagged, and it is a `.tk` domain. It is
exactly the row you care about.

In [ ]:
print(df.isna().sum())

In [ ]:
young = df[df["age_days"] < 30]

print(young[["host", "age_days", "flagged"]])
print()
print("rows returned:", len(young))

Two rows. `acme-login.tk` is not one of them.

A comparison against a missing value is false, not an error. The filter did not
fail. It quietly answered a slightly different question than the one you asked.

An average over the same column ignores the gap instead of poisoning the result,
which is a different choice again.

In [ ]:
print("mean: ", df["age_days"].mean())
print("count:", df["age_days"].count())
print("len:  ", len(df))

Neither behaviour is wrong and they are not the same behaviour. Find the gaps
first, then decide per column whether to drop, fill, or treat missing as its own
category.

In [ ]:
# Treating missing as its own category, which is often the honest option
labelled = df["age_days"].isna().map({True: "unknown", False: "known"})
print(labelled.value_counts())

## 5. Writes that do not land

This looks like it sets a value.

In [ ]:
import warnings

test = df.copy()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    test[test["tld"] == "tk"]["flagged"] = False
    print("warnings raised:", [w.category.__name__ for w in caught])

print("flagged values for tk:", test.loc[test["tld"] == "tk", "flagged"].tolist())

Nothing changed. `test[test["tld"] == "tk"]` handed back a new object and the
assignment landed on that, not on `test`. This is lesson 1.2 in a different
costume: two steps, and the second one hit the wrong thing.

pandas warns about it, and from pandas 3.0 copy-on-write is the default and
chained assignment never works at all.

Say it in one step.

In [ ]:
fixed = df.copy()
fixed.loc[fixed["tld"] == "tk", "flagged"] = False

print("flagged values for tk:", fixed.loc[fixed["tld"] == "tk", "flagged"].tolist())

The habit: if you are about to write into something you just selected, `.loc` it.

## 6. Your turn

You are enriching the host table with owner information. The merge runs cleanly
and gives you more rows than you started with.

Change one thing so the row count is unchanged.

In [ ]:
owners = pd.DataFrame({
    "host":  ["login.acme.co", "cdn.acme.co", "cdn.acme.co", "mail.acme.co"],
    "owner": ["it", "web", "web-team", "it"],
})


def enrich(hosts, owners):
    # TODO: one change so no rows are invented
    return hosts.merge(owners, on="host", how="left")


out = enrich(df, owners)

print("before:", len(df), " after:", len(out))
print("row count unchanged?", len(out) == len(df))
print()
print(out[["host", "owner"]])

Look at `owners` and ask whether `host` is unique in it.

<details>
<summary>Answer</summary>

`cdn.acme.co` appears twice in `owners`, so the left join produces one output row
per match and the table grows.

```python
def enrich(hosts, owners):
    return hosts.merge(owners.drop_duplicates(subset="host"), on="host", how="left")
```

Better still, make pandas refuse rather than let it happen quietly:

```python
hosts.merge(owners, on="host", how="left", validate="many_to_one")
```

That raises `MergeError` when the right-hand key is not unique, which turns a
silent data problem into a loud one.

</details>

## What you now have

- `dtypes` after every load, because one stray value turns a column into text
- A list inside the brackets picks columns, a condition picks rows
- Grouping is two calls: sort into buckets, then say what to work out per bucket
- A comparison against a missing value is false, while an average skips it
- `.loc` for anything you are about to write into
- `len()` before and after every merge

Next is lesson 1.5, SQL, because most security data lives in a database or a SIEM rather than a CSV.